In [ ]:
import sys
import os
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
# PDF Loader - PDF를 마크다운으로 변환

import os
from pathlib import Path
from typing import List, Optional

def table_to_markdown(table: List[List]) -> str:
    """
    표 데이터를 마크다운 테이블 형식으로 변환하는 헬퍼 함수
    
    Args:
        table: 2차원 리스트 형태의 표 데이터
    
    Returns:
        마크다운 테이블 문자열
    """
    if not table or len(table) == 0:
        return ""
    
    # 빈 셀을 빈 문자열로 변환
    def clean_cell(cell):
        if cell is None:
            return ""
        return str(cell).strip()
    
    # 표 데이터 정리
    cleaned_table = [[clean_cell(cell) for cell in row] for row in table]
    
    # 최대 컬럼 수 확인
    max_cols = max(len(row) for row in cleaned_table) if cleaned_table else 0
    
    # 모든 행을 동일한 컬럼 수로 맞춤
    normalized_table = []
    for row in cleaned_table:
        normalized_row = row + [""] * (max_cols - len(row))
        normalized_table.append(normalized_row)
    
    if not normalized_table:
        return ""
    
    markdown_lines = []
    
    # 헤더 행 (첫 번째 행)
    header = normalized_table[0]
    markdown_lines.append("| " + " | ".join(header) + " |")
    
    # 구분선
    markdown_lines.append("| " + " | ".join(["---"] * len(header)) + " |")
    
    # 데이터 행들
    for row in normalized_table[1:]:
        markdown_lines.append("| " + " | ".join(row) + " |")
    
    return "\n".join(markdown_lines)


def extract_text_from_pdf(pdf_path: str, password: str = None) -> str:
    """
    PDF 파일을 마크다운 형식으로 변환하여 반환하는 함수
    
    표는 마크다운 테이블 형식으로 변환되고, 텍스트는 그대로 유지됩니다.
    
    Args:
        pdf_path: PDF 파일 경로 (상대 경로 또는 절대 경로)
        password: 암호화된 PDF의 비밀번호 (선택사항)
    
    Returns:
        마크다운 형식으로 변환된 문자열 (텍스트 + 표)
    
    Raises:
        FileNotFoundError: PDF 파일을 찾을 수 없을 때
        ImportError: 필요한 PDF 라이브러리가 설치되지 않았을 때
        Exception: 암호가 틀렸거나 PDF를 읽을 수 없을 때
    """
    # 파일 경로 확인 및 절대 경로로 변환
    pdf_path = Path(pdf_path)
    if not pdf_path.is_absolute():
        # 노트북 위치 기준 상대 경로 처리
        # 노트북은 asset_ai_portal/tests 폴더에 있고, documents는 20_code_test 루트에 있음
        current_dir = Path.cwd()
        
        # asset_ai_portal/tests에서 실행 중이면 상위로 두 번 이동 (20_code_test 루트)
        if current_dir.name == 'tests' and current_dir.parent.name == 'asset_ai_portal':
            project_root = current_dir.parent.parent  # tests -> asset_ai_portal -> 20_code_test
        elif current_dir.name == 'asset_ai_portal':
            project_root = current_dir.parent  # asset_ai_portal -> 20_code_test
        else:
            # 20_code_test에서 실행 중이면 그대로 사용
            project_root = current_dir
        
        pdf_path = project_root / pdf_path
    
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {pdf_path}")
    
    # 여러 PDF 라이브러리 시도 (우선순위 순)
    # 1. pdfplumber (표 추출에 유리, 마크다운 변환에 최적)
    try:
        import pdfplumber
        
        markdown_parts = []
        with pdfplumber.open(str(pdf_path), password=password) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                page_content = []
                
                # 페이지 번호 추가 (선택사항)
                # page_content.append(f"\n## 페이지 {page_num}\n")
                
                # 표 추출 (표가 있으면 먼저 표를 추출)
                tables = page.extract_tables()
                if tables:
                    for table_idx, table in enumerate(tables):
                        if table:
                            markdown_table = table_to_markdown(table)
                            if markdown_table:
                                page_content.append(markdown_table)
                                page_content.append("")  # 표 다음에 빈 줄 추가
                
                # 텍스트 추출
                text = page.extract_text()
                if text:
                    # 표와 겹치는 텍스트를 제거하기 위해 간단한 필터링
                    # (실제로는 더 정교한 로직이 필요할 수 있음)
                    page_content.append(text)
                
                if page_content:
                    markdown_parts.append("\n".join(page_content))
        
        return "\n\n".join(markdown_parts) if markdown_parts else ""
        
    except ImportError:
        pass
    except Exception as e:
        # 암호 오류 등 다른 에러는 다음 라이브러리로 시도
        if 'password' in str(e).lower() or 'encrypted' in str(e).lower():
            raise ValueError(f"PDF 암호가 올바르지 않거나 암호화된 PDF를 읽을 수 없습니다: {e}")
        pass

    # 모든 라이브러리가 없으면 에러
    raise ImportError(
        "PDF 텍스트 추출을 위한 라이브러리가 설치되지 않았습니다. "
        "설치 명령: pip install pdfplumber\n"
        "표 추출 기능을 사용하려면 pdfplumber를 설치하는 것을 권장합니다."
    )


In [ ]:
# Docling Loader - PDF를 마크다운으로 변환

import os
from pathlib import Path
from typing import Optional

# TESSDATA_PREFIX 환경 변수를 모듈 레벨에서 설정
# docling이 import될 때 tesserocr를 초기화할 수 있으므로 미리 설정
_tessdata_path = os.environ.get('TESSDATA_PREFIX', '/opt/homebrew/share/tessdata')
if not os.environ.get('TESSDATA_PREFIX'):
    os.environ['TESSDATA_PREFIX'] = _tessdata_path

# tesserocr를 직접 import하여 환경 변수를 확인하도록 강제
# docling이 내부적으로 tesserocr.get_languages()를 호출할 때 환경 변수를 읽을 수 있도록
try:
    import tesserocr
    # tesserocr를 초기화하여 환경 변수를 확인하도록 강제
    # get_languages()가 환경 변수를 읽지 못하는 문제를 해결하기 위해
    # PyTessBaseAPI를 사용하여 초기화
    _test_api = tesserocr.PyTessBaseAPI(path=_tessdata_path)
    _test_api.End()
except ImportError:
    # tesserocr가 설치되지 않은 경우 무시 (나중에 오류 처리됨)
    pass
except Exception:
    # 초기화 실패는 무시 (나중에 오류 처리됨)
    pass

from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption

# Tesseract OCR과 CPU를 명시적으로 지정하는 설정
# PdfFormatOption의 pipeline_options를 통해 ThreadedPdfPipelineOptions 전달
# ThreadedPdfPipelineOptions의 ocr_options에 TesseractOcrOptions 지정
from docling.datamodel.pipeline_options import (
    ThreadedPdfPipelineOptions,
    TesseractOcrOptions
)
from docling.datamodel.accelerator_options import AcceleratorOptions

# 암호화된 PDF를 다른 라이브러리로 읽어서 암호를 해제하고 임시 파일로 저장
import tempfile

def extract_text_from_pdf_with_docling(pdf_path: str, password: str = None) -> str:
    """
    Docling 라이브러리를 사용하여 PDF 파일을 마크다운 형식으로 변환하는 함수
    
    Docling은 IBM에서 개발한 문서 변환 라이브러리로, 표, 레이아웃, 구조를 잘 보존하며
    마크다운으로 변환합니다.
    
    Tesseract OCR과 CPU 명시적 지정:
    - OCR 모델: Tesseract (PdfFormatOption의 pipeline_options를 통해 명시적으로 지정)
    - 가속기: CPU (AcceleratorOptions를 통해 명시적으로 지정)
    - 암호화된 PDF: PyPDF2/pypdf로 암호 해제 후 변환
    
    설정 방법:
    - PdfFormatOption의 pipeline_options 파라미터에 ThreadedPdfPipelineOptions 전달
    - ThreadedPdfPipelineOptions의 ocr_options에 TesseractOcrOptions 지정
    - ThreadedPdfPipelineOptions의 accelerator_options에 AcceleratorOptions(device='cpu') 지정
    - sys.platform 변경 없이 공식 API를 통해 명시적으로 설정
    
    주의사항:
    - TesseractOcrOptions를 사용하려면 tesserocr 라이브러리와 올바른 설정이 필요합니다.
    - Tesseract OCR 설정 오류 시 폴백 없이 오류가 발생하여 프로세스가 중지됩니다.
    - AcceleratorOptions(device='cpu')를 통해 CPU 사용이 명시적으로 지정됩니다.
    - TESSDATA_PREFIX 환경 변수가 올바르게 설정되어 있어야 합니다.
    
    Args:
        pdf_path: PDF 파일 경로 (상대 경로 또는 절대 경로)
        password: 암호화된 PDF의 비밀번호 (선택사항)
    
    Returns:
        마크다운 형식으로 변환된 문자열
    
    Raises:
        FileNotFoundError: PDF 파일을 찾을 수 없을 때
        ImportError: docling 라이브러리가 설치되지 않았을 때
        Exception: PDF를 읽을 수 없을 때
    """
    # TESSDATA_PREFIX 환경 변수 확인 및 설정
    # 모듈 레벨에서 이미 설정되었지만, 함수 내에서도 재확인
    tessdata_path = os.environ.get('TESSDATA_PREFIX', '/opt/homebrew/share/tessdata')
    if not os.environ.get('TESSDATA_PREFIX'):
        os.environ['TESSDATA_PREFIX'] = tessdata_path
    
    # 파일 경로 확인 및 절대 경로로 변환
    pdf_path = Path(pdf_path)
    if not pdf_path.is_absolute():
        # 노트북 위치 기준 상대 경로 처리
        # 노트북은 asset_ai_portal/tests 폴더에 있고, documents는 20_code_test 루트에 있음
        current_dir = Path.cwd()
        
        # asset_ai_portal/tests에서 실행 중이면 상위로 두 번 이동 (20_code_test 루트)
        if current_dir.name == 'tests' and current_dir.parent.name == 'asset_ai_portal':
            project_root = current_dir.parent.parent  # tests -> asset_ai_portal -> 20_code_test
        elif current_dir.name == 'asset_ai_portal':
            project_root = current_dir.parent  # asset_ai_portal -> 20_code_test
        else:
            # 20_code_test에서 실행 중이면 그대로 사용
            project_root = current_dir
        
        pdf_path = project_root / pdf_path
    
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {pdf_path}")
    
    # 암호화된 PDF 처리: 다른 라이브러리로 암호 해제 후 docling으로 변환
    temp_pdf_path = None
    try:
        if password:
            
            
            # PyPDF2 또는 pypdf를 사용하여 암호화된 PDF를 읽고 암호 해제된 PDF로 저장
            try:
                from PyPDF2 import PdfWriter, PdfReader
                
                reader = PdfReader(str(pdf_path))
                if reader.is_encrypted:
                    if not reader.decrypt(password):
                        raise ValueError("PDF 암호가 올바르지 않습니다.")
                
                writer = PdfWriter()
                for page in reader.pages:
                    writer.add_page(page)
                
                # 임시 파일 생성
                temp_fd, temp_pdf_path = tempfile.mkstemp(suffix='.pdf')
                os.close(temp_fd)
                
                with open(temp_pdf_path, 'wb') as temp_file:
                    writer.write(temp_file)
                
                pdf_path_to_convert = temp_pdf_path
                
            except ImportError:
                # PyPDF2가 없으면 pypdf 시도
                try:
                    from pypdf import PdfWriter, PdfReader
                    
                    reader = PdfReader(str(pdf_path), password=password)
                    writer = PdfWriter()
                    for page in reader.pages:
                        writer.add_page(page)
                    
                    # 임시 파일 생성
                    temp_fd, temp_pdf_path = tempfile.mkstemp(suffix='.pdf')
                    os.close(temp_fd)
                    
                    with open(temp_pdf_path, 'wb') as temp_file:
                        writer.write(temp_file)
                    
                    pdf_path_to_convert = temp_pdf_path
                    
                except ImportError:
                    raise ImportError(
                        "암호화된 PDF를 처리하기 위해 PyPDF2 또는 pypdf가 필요합니다.\n"
                        "설치 명령: pip install PyPDF2 또는 pip install pypdf"
                    )
        else:
            # 암호화되지 않은 PDF는 원본 경로 사용
            pdf_path_to_convert = str(pdf_path)
        
        # Docling 라이브러리 사용
        # Tesseract OCR과 CPU를 명시적으로 지정하여 사용
        
        # TESSDATA_PREFIX 환경 변수 설정 (tesserocr 사용 시 필요)
        tessdata_path = os.environ.get('TESSDATA_PREFIX', '/opt/homebrew/share/tessdata')
        if not os.environ.get('TESSDATA_PREFIX'):
            os.environ['TESSDATA_PREFIX'] = tessdata_path       
        
        # Tesseract OCR 옵션 생성 (실패 시 오류 발생, 폴백 없음)
        # TesseractOcrOptions를 사용하려면 tesserocr 라이브러리와 올바른 설정이 필요합니다.
        tesseract_ocr_opts = TesseractOcrOptions(
            lang=['eng', 'kor'],  # 영어와 한국어 지원
            bitmap_area_threshold=0.01,
            force_full_page_ocr=False,
            path=tessdata_path  # tessdata 경로 명시
        )
        
        # ThreadedPdfPipelineOptions 생성 (Tesseract OCR과 CPU 지정)
        threaded_pipeline_opts = ThreadedPdfPipelineOptions(
            ocr_options=tesseract_ocr_opts,
            accelerator_options=AcceleratorOptions(device='cpu')
        )
        
        # PdfFormatOption에 pipeline_options 전달
        pdf_option = PdfFormatOption(
            pipeline_options=threaded_pipeline_opts
        )
        
        converter = DocumentConverter(
            allowed_formats=[InputFormat.PDF],
            format_options={
                InputFormat.PDF: pdf_option
            }
        )
        
        print("ℹ️ Tesseract OCR과 CPU가 pipeline_options를 통해 명시적으로 지정되었습니다.")
        
        # converter.convert() 호출 전에 TESSDATA_PREFIX 환경 변수 재확인 및 설정
        # pipeline 초기화 시점에 tesserocr가 환경 변수를 확인하므로 명시적으로 설정
        if not os.environ.get('TESSDATA_PREFIX'):
            os.environ['TESSDATA_PREFIX'] = tessdata_path
        print(f"   TESSDATA_PREFIX: {os.environ.get('TESSDATA_PREFIX')}")

        # PDF 변환 (암호 해제된 PDF 또는 원본 PDF)
        # Tesseract OCR 실패 시 오류 발생 (폴백 없음)
        result = converter.convert(pdf_path_to_convert)
        
        # 마크다운으로 내보내기
        markdown_content = result.document.export_to_markdown()
        
        return markdown_content
        
    except ImportError as import_err:
        # ImportError 처리 (Tesseract OCR 설정 오류 포함)
        error_msg = str(import_err)
        
        if 'docling' in error_msg.lower() and 'tesserocr' not in error_msg.lower():
            # docling import 실패
            raise ImportError(
                f"Docling 라이브러리가 설치되지 않았습니다.\n"
                f"오류 메시지: {error_msg}\n"
                f"설치 명령: pip install docling\n"
                f"또는: pip install 'docling[pdf]'"
            )
        elif 'tesserocr' in error_msg.lower():
            # Tesseract OCR 설정 오류 - 폴백 없이 오류 발생
            tessdata_path_for_error = os.environ.get('TESSDATA_PREFIX', '/opt/homebrew/share/tessdata')
            raise ImportError(
                f"Tesseract OCR 설정 오류가 발생했습니다.\n"
                f"오류 메시지: {error_msg}\n"
                f"\n"
                f"해결 방법:\n"
                f"1. tesserocr가 설치되어 있는지 확인: pip install tesserocr\n"
                f"2. TESSDATA_PREFIX 환경 변수가 올바르게 설정되어 있는지 확인\n"
                f"3. tessdata 경로에 언어 모델 파일이 있는지 확인: {tessdata_path_for_error}\n"
                f"4. 한글 언어 모델 확인: {tessdata_path_for_error}/kor.traineddata"
            )
        else:
            # 다른 ImportError는 그대로 전파
            raise
    except Exception as e:
        # 모든 오류를 명확하게 출력
        error_msg = str(e)
        error_type = type(e).__name__
        import traceback
        
        print(f"❌ Docling PDF 변환 오류 발생")
        print(f"   오류 유형: {error_type}")
        print(f"   오류 메시지: {error_msg}")
        print(f"\n   전체 스택 트레이스:")
        traceback.print_exc()
        
        # 암호화된 PDF 관련 오류인지 확인
        # password가 제공되었는데 오류가 발생하면 암호화된 PDF 처리 실패로 간주
        if password:
            # 암호화 관련 키워드 확인
            is_password_error = (
                'password' in error_msg.lower() or 
                'encrypted' in error_msg.lower() or 
                'incorrect password' in error_msg.lower() or
                'not valid' in error_msg.lower() or
                'inconsistent number of pages' in error_msg.lower() or
                error_type == 'ConversionError'
            )
            
            if is_password_error:
                raise ValueError(
                    f"Docling으로 암호화된 PDF를 처리할 수 없습니다.\n"
                    f"오류 유형: {error_type}\n"
                    f"오류 메시지: {error_msg}\n"
                    f"암호화된 PDF는 extract_text_from_pdf() 함수를 사용해주세요."
                )
        
        # 기타 오류
        raise Exception(
            f"PDF 변환 중 오류가 발생했습니다.\n"
            f"오류 유형: {error_type}\n"
            f"오류 메시지: {error_msg}"
        )
    finally:
        # 임시 파일 정리
        if temp_pdf_path and os.path.exists(temp_pdf_path):
            try:
                os.remove(temp_pdf_path)
            except Exception:
                pass  # 임시 파일 삭제 실패는 무시


# 사용 예시
# pdf_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# try:
#     markdown_text = extract_text_from_pdf_with_docling(pdf_file_path)
#     print(f"✅ Docling으로 PDF 마크다운 변환 완료 ({len(markdown_text)} 문자)")
#     print("\n" + "="*80)
#     print("변환된 마크다운 (처음 1000자):")
#     print("="*80)
#     print(markdown_text[:1000])
#     if len(markdown_text) > 1000:
#         print(f"\n... (총 {len(markdown_text)} 문자 중 처음 1000자만 표시)")
# except Exception as e:
#     print(f"❌ 오류 발생: {e}")


In [ ]:
# excel loader

import os
from pathlib import Path
from typing import Dict, List

def extract_text_from_excel(excel_path: str) -> Dict[str, str]:
    """
    Excel 파일(.xlsx, .xls)에서 모든 시트의 텍스트를 추출하는 함수
    
    Args:
        excel_path: Excel 파일 경로 (상대 경로 또는 절대 경로)
    
    Returns:
        시트 이름을 키로 하고 추출된 텍스트를 값으로 하는 딕셔너리
    
    Raises:
        FileNotFoundError: Excel 파일을 찾을 수 없을 때
        ImportError: 필요한 Excel 라이브러리가 설치되지 않았을 때
    """
    # 파일 경로 확인 및 절대 경로로 변환
    excel_path = Path(excel_path)
    if not excel_path.is_absolute():
        # 노트북 위치 기준 상대 경로 처리
        # 노트북은 asset_ai_portal/tests 폴더에 있고, documents는 20_code_test 루트에 있음
        current_dir = Path.cwd()
        
        # asset_ai_portal/tests에서 실행 중이면 상위로 두 번 이동 (20_code_test 루트)
        if current_dir.name == 'tests' and current_dir.parent.name == 'asset_ai_portal':
            project_root = current_dir.parent.parent  # tests -> asset_ai_portal -> 20_code_test
        elif current_dir.name == 'asset_ai_portal':
            project_root = current_dir.parent  # asset_ai_portal -> 20_code_test
        else:
            # 20_code_test에서 실행 중이면 그대로 사용
            project_root = current_dir
        
        excel_path = project_root / excel_path
    
    if not excel_path.exists():
        raise FileNotFoundError(f"Excel 파일을 찾을 수 없습니다: {excel_path}")
    
    # 파일 확장자 확인
    file_ext = excel_path.suffix.lower()
    
    # 여러 Excel 라이브러리 시도 (우선순위 순)
    # 1. pandas + openpyxl/xlrd (가장 편리함)
    try:
        import pandas as pd
        
        # 모든 시트 읽기
        if file_ext == '.xlsx':
            excel_file = pd.ExcelFile(str(excel_path), engine='openpyxl')
        elif file_ext == '.xls':
            excel_file = pd.ExcelFile(str(excel_path), engine='xlrd')
        else:
            # 자동 감지
            excel_file = pd.ExcelFile(str(excel_path))
        
        sheets_text = {}
        for sheet_name in excel_file.sheet_names:
            df = pd.read_excel(excel_file, sheet_name=sheet_name)
            # DataFrame을 텍스트로 변환
            text_parts = []
            # 헤더 포함하여 모든 셀의 값을 문자열로 변환
            for idx, row in df.iterrows():
                row_values = [str(val) if pd.notna(val) else '' for val in row.values]
                text_parts.append(' | '.join(row_values))
            
            sheets_text[sheet_name] = '\n'.join(text_parts)
        
        return sheets_text
    except ImportError as e:
        if 'pandas' in str(e):
            pass  # pandas가 없으면 다음 방법 시도
        elif 'openpyxl' in str(e) or 'xlrd' in str(e):
            # pandas는 있지만 엔진이 없는 경우
            raise ImportError(
                f"Excel 파일을 읽기 위한 엔진이 필요합니다.\n"
                f".xlsx 파일: pip install openpyxl\n"
                f".xls 파일: pip install xlrd"
            )
        else:
            raise
    
    # 2. openpyxl (xlsx 파일용)
    if file_ext == '.xlsx':
        try:
            from openpyxl import load_workbook
            
            workbook = load_workbook(str(excel_path), data_only=True)
            sheets_text = {}
            
            for sheet_name in workbook.sheetnames:
                sheet = workbook[sheet_name]
                text_parts = []
                
                for row in sheet.iter_rows(values_only=True):
                    row_values = [str(val) if val is not None else '' for val in row]
                    text_parts.append(' | '.join(row_values))
                
                sheets_text[sheet_name] = '\n'.join(text_parts)
            
            return sheets_text
        except ImportError:
            pass
    
    # 3. xlrd (xls 파일용)
    if file_ext == '.xls':
        try:
            import xlrd
            
            workbook = xlrd.open_workbook(str(excel_path))
            sheets_text = {}
            
            for sheet_name in workbook.sheet_names():
                sheet = workbook.sheet_by_name(sheet_name)
                text_parts = []
                
                for row_idx in range(sheet.nrows):
                    row_values = [str(sheet.cell_value(row_idx, col_idx)) 
                                 for col_idx in range(sheet.ncols)]
                    text_parts.append(' | '.join(row_values))
                
                sheets_text[sheet_name] = '\n'.join(text_parts)
            
            return sheets_text
        except ImportError:
            pass
    
    # 모든 라이브러리가 없으면 에러
    raise ImportError(
        "Excel 텍스트 추출을 위한 라이브러리가 설치되지 않았습니다.\n"
        "다음 중 하나를 설치해주세요:\n"
        "  - pandas + openpyxl (권장): pip install pandas openpyxl\n"
        "  - pandas + xlrd (.xls 파일용): pip install pandas xlrd\n"
        "  - openpyxl (.xlsx 파일용): pip install openpyxl\n"
        "  - xlrd (.xls 파일용): pip install xlrd"
    )


def get_all_sheets_text(excel_path: str) -> str:
    """
    Excel 파일의 모든 시트 텍스트를 하나의 문자열로 반환하는 편의 함수
    
    Args:
        excel_path: Excel 파일 경로
    
    Returns:
        모든 시트의 텍스트를 합친 문자열
    """
    sheets_dict = extract_text_from_excel(excel_path)
    
    result_parts = []
    for sheet_name, sheet_text in sheets_dict.items():
        result_parts.append(f"=== 시트: {sheet_name} ===")
        result_parts.append(sheet_text)
        result_parts.append("")  # 빈 줄 추가
    
    return '\n'.join(result_parts)


In [ ]:
# Document Loader - 파일 형식에 따라 적절한 함수 호출

from pathlib import Path
from typing import Union

def load_document(file_path: str, password: str = None) -> str:
    """
    파일 형식에 따라 적절한 텍스트 추출 함수를 호출하여 텍스트를 반환하는 통합 함수
    
    지원 형식:
    - PDF: .pdf 파일 (암호화된 PDF 지원)
    - Excel: .xlsx, .xls 파일
    
    Args:
        file_path: 문서 파일 경로 (상대 경로 또는 절대 경로)
        password: PDF 파일이 암호화된 경우 비밀번호 (선택사항)
    
    Returns:
        추출된 텍스트 문자열
        - PDF: 전체 텍스트
        - Excel: 모든 시트의 텍스트를 합친 문자열
    
    Raises:
        FileNotFoundError: 파일을 찾을 수 없을 때
        ValueError: 지원하지 않는 파일 형식일 때 또는 PDF 암호가 틀렸을 때
        ImportError: 필요한 라이브러리가 설치되지 않았을 때
    """
    file_path_obj = Path(file_path)
    file_ext = file_path_obj.suffix.lower()
    
    # 파일 형식에 따라 적절한 함수 호출
    if file_ext == '.pdf':
        # PDF 파일 처리 (암호 전달)
        # return extract_text_from_pdf(file_path, password=password)
        return extract_text_from_pdf_with_docling(file_path, password)
    
    elif file_ext in ['.xlsx', '.xls']:
        # Excel 파일 처리 - 모든 시트의 텍스트를 하나의 문자열로 반환
        return get_all_sheets_text(file_path)
    
    else:
        raise ValueError(
            f"지원하지 않는 파일 형식입니다: {file_ext}\n"
            f"지원 형식: .pdf, .xlsx, .xls"
        )




In [87]:
# text 추출

# 사용 예시
test_files = [
    # "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx",
    "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf",
    # "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf",
    # "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/대신.pdf"
]
_password = None
_password = '345678'
for file_path in test_files:
    try:
        print(f"\n{'='*80}")
        print(f"📄 파일: {Path(file_path).name} ({Path(file_path).suffix})")
        print('='*80)
        
        document_text = load_document(file_path, _password)
        
        print(f"✅ 문서 텍스트 추출 완료")
        print(f"   - 총 문자 수: {len(document_text)}")
        print(f"   {'-'*76}")
        print(f"   {document_text}...")
        
        # 마지막 파일의 텍스트를 document_text 변수에 저장
        if file_path == test_files[-1]:
            document_text = document_text
            
    except Exception as e:
        print(f"❌ 오류 발생 ({Path(file_path).name}): {e}")

2025-12-30 11:25:12,578 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-30 11:25:12,579 - INFO - Going to convert document batch...
2025-12-30 11:25:12,579 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 53af76e463a559e5094ece08e7d8124b
2025-12-30 11:25:12,656 - INFO - Accelerator device: 'cpu'



📄 파일: 신한라이프_251127.pdf (.pdf)
ℹ️ Tesseract OCR과 CPU가 pipeline_options를 통해 명시적으로 지정되었습니다.
   TESSDATA_PREFIX: /opt/homebrew/share/tessdata


2025-12-30 11:25:13,393 - INFO - Accelerator device: 'cpu'
2025-12-30 11:25:13,955 - INFO - Processing document tmpwpoagjmv.pdf
2025-12-30 11:25:16,912 - INFO - Finished converting document tmpwpoagjmv.pdf in 4.33 sec.


✅ 문서 텍스트 추출 완료
   - 총 문자 수: 3081
   ----------------------------------------------------------------------------
   ## 변액 펀드별 설정/해지 내역

기준일자 : 2025-11-27

수신처 : 삼성자산운용

CUTOFF대상여부 : N

| 통합 펀드코드   | 서브 펀드코드   | 펀드명                   | 운용사       | 입금액       | 출금액       | 당일이체좌수   | 당일이체금액   | 이체예정금액   |             |             |                |
|-----------------|-----------------|--------------------------|--------------|--------------|--------------|----------------|----------------|----------------|-------------|-------------|----------------|
|                 |                 |                          |              |              |              |                |                | 2025-11-28     | 2025-12-01  | 2025-12-02  | 2025-12-03     |
| V0043           |                 | 글로벌멀티에셋자산배분형 | 삼성자산운용 | 8,251,635    | 16,627,375   | -5,578,407     | -8,375,740     | 451,845        | -1,858,420  | -12,733,690 | 1,984,387      |
| V0058           |                 | 글로벌헷지펀드배분형   

In [88]:
# LLM 모델 정의

LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")
LLM_TEMPERATURE=os.getenv("LLM_TEMPERATURE")

# vLLM 모델 인스턴스 생성
llm = init_chat_model(
    "openai:",
    temperature=LLM_TEMPERATURE,
    top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.9로 설정
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY
)

In [89]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

_gettering_data = None
if document_text:
    # 메시지 객체 생성
    system_msg = SystemMessage("당신은 자산운용사에서 변액일임펀드 설정/해지 업무를 담당하는 오퍼레이터 입니다.")
    human_msg = HumanMessage(f"""
    아래는 수익자가 보내온 변액일임펀드 설정/해지 지시서 메일입니다.
    think step by step, 주어진 메일 내용을 분석하여 시스템에 입력할 데이터를 수집하세요.

    ### 변액일임펀드 설정/해지 지시서 메일 내용 ###
    {document_text}

    ** 반드시 지켜야 할 중요 지침 **
    1. 모든 종목을 전부 수집하세요.(주요 종목만 수집하면 안됩니다.)
    2. 확정분과 청구분을 구분하는 기준을 명확히 정의하세요.
    3. 2번 지침에서 정의한 기준에 따라 확정분과 청구분으로 구분하세요.
    4. 금액(amount)과 좌수(unit)를 구분하세요.
    5. 날짜 정보는 모두 수집하세요.
    6. 펀드별로 데이터를 정리하세요.
    7. 추측과 예상을 하지 말고 사실만 출력하세요.
    8. 수집 결과의 오류 여부를 검증하고 오류가 있으면 수정하세요.    
    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]
    response = llm.invoke(messages)  # AIMessage 반환
    # print(response)
    _gettering_data = response.content

2025-12-30 11:26:04,849 - INFO - HTTP Request: POST http://localhost:3900/v1/chat/completions "HTTP/1.1 200 OK"


In [90]:
from IPython.display import Markdown, display

def display_markdown(response):
    # LLM 응답을 마크다운 형식으로 보기 좋게 표시
    if 'response' in locals():
        display(Markdown(response.content))
        
        # 추가 정보 (토큰 사용량 등)를 표시
        if hasattr(response, 'response_metadata') and response.response_metadata:
            metadata = response.response_metadata
            if 'token_usage' in metadata:
                print("\n---")
                print("**토큰 사용량:**")
                print(f"- 입력 토큰: {metadata['token_usage'].get('prompt_tokens', 'N/A')}")
                print(f"- 출력 토큰: {metadata['token_usage'].get('completion_tokens', 'N/A')}")
                print(f"- 총 토큰: {metadata['token_usage'].get('total_tokens', 'N/A')}")
    else:
        print("⚠️ 'response' 변수를 찾을 수 없습니다. 먼저 LLM을 호출해주세요.")

display_markdown(response)

주어진 메일 내용을 **step by step** 분석하여, 시스템에 입력할 데이터를 **사실만** 기반으로 수집하고, **오류 검증** 및 **구분 기준 명확화**를 수행합니다.

---

### ✅ **1. 수집 대상 정의 (지침 1, 6, 7)**  
- **모든 종목** 수집: 6개 통합펀드 (V0043, V0058, VU51000, VU71001, VU71002, VU80006)  
- **각 펀드별로 데이터 정리**  
- **추측 없이, 표에 명시된 값만 사용**  
- **보수 항목(운영보수, 투자일임보수 등)**은 **서브펀드 이체금액에 포함되지 않음** (※ 주석 참조) → **이 항목들은 본 메일에 실제 금액이 없음** → 수집 불가

---

### ✅ **2. 확정분 vs 청구분 구분 기준 (지침 2, 3)**  
> **기준 정의**:  
> - **확정분 (Confirmed)**: **기준일자(2025-11-27) 기준으로 당일 발생한 실제 이체 내역**  
>   → **당일이체좌수**, **당일이체금액**  
> - **청구분 (Requested / Scheduled)**: **미래 이체 예정일에 대한 예정 금액**  
>   → **이체예정금액** 열에 명시된 **2025-11-28, 2025-12-01, 2025-12-02, 2025-12-03**의 금액  

> ✅ **근거**:  
> - 메일 상단에 "**기준일자 : 2025-11-27**" 명시  
> - "**서브펀드의 이체금액은 NAV기준으로 차감되는 보수 제외된 금액**" → 이는 **이체금액의 구성**에 대한 설명이지, **확정/청구 구분**에 대한 설명 아님  
> - "**이체예정금액**" 열은 **미래 날짜**로 명시 → **청구분**  
> - "**당일이체좌수/금액**"은 **기준일자 당일** → **확정분**  

> ✅ **결론**:  
> - **확정분** = **당일이체좌수**, **당일이체금액** (2025-11-27 기준)  
> - **청구분** = **이체예정금액** 열의 4개 날짜(2025-11-28, 12-01, 12-02, 12-03)  

---

### ✅ **3. 금액(amount) vs 좌수(unit) 구분 (지침 4)**  
- **당일이체좌수** → **좌수(unit)**  
- **당일이체금액**, **입금액**, **출금액**, **이체예정금액** → **금액(amount)**  
- **좌수**는 **당일이체좌수** 열에만 존재 → 그 외는 모두 금액  

---

### ✅ **4. 날짜 정보 수집 (지침 5)**  
- 기준일자: **2025-11-27**  
- 이체예정일: **2025-11-28, 2025-12-01, 2025-12-02, 2025-12-03**  
- 메일 발신일/승인일은 **업무 기록용**이므로 **시스템 입력 대상 아님** (지침 1~8에 명시되지 않음)  

---

### ✅ **5. 펀드별 데이터 수집 (지침 6)**  
각 펀드에 대해 다음 항목 수집:  
- 통합펀드코드  
- 펀드명  
- 운용사  
- 입금액 (amount)  
- 출금액 (amount)  
- 당일이체좌수 (unit)  
- 당일이체금액 (amount)  
- 이체예정금액 (amount) → 4개 날짜별로 분리  

> **※ 서브펀드코드는 모두 공백 → 수집 불가 → "N/A"로 표기**  
> **※ 합계 행은 펀드별 데이터가 아님 → 제외**  
> **※ 보수/감사인/선급법인세 등은 실제 금액 없음 → 수집 불가**  

---

### ✅ **6. 오류 검증 및 수정 (지침 8)**  
- **입금액/출금액**과 **당일이체금액**의 관계 확인:  
  - 당일이체금액 = 입금액 - 출금액?  
    - V0043: 8,251,635 - 16,627,375 = **-8,375,740** → ✅ 일치  
    - V0058: 23,358,300 - 73,058,898 = **-49,700,598** → ✅ 일치  
    - VU51000: 14,174,090 - 6,460,559 = **7,713,531** → ✅ 일치  
    - VU71001: 69,636,310 - 2,765,037 = **66,871,273** → ✅ 일치  
    - VU71002: 53,535,858 - 1,120,424 = **52,415,434** → ✅ 일치  
    - VU80006: 23,323,440 - 16,521,582 = **6,801,858** → ✅ 일치  

→ **당일이체금액 = 입금액 - 출금액** → **데이터 일관성 확인 완료**  

- **당일이체좌수**는 **입금/출금과 직접 연관 없음** → 별도 항목으로 유지  

- **이체예정금액** 열의 합계와 총계 비교:  
  - 2025-11-28: 56,607,450 vs 총계 41,284,077 → **다름** → 정상 (합계는 펀드별 합, 총계는 확정+청구 합)  
  - 총계는 **당일이체 + 이체예정**의 합이 아님 → **표의 총계는 단순 합계** → 오류 아님  

→ **모든 데이터 일관성 확인 완료, 오류 없음**

---

### ✅ **7. 최종 시스템 입력 데이터 (사실만, 펀드별 정리)**

#### 📌 **V0043 - 글로벌멀티에셋자산배분형**
- 통합펀드코드: V0043  
- 서브펀드코드: N/A  
- 펀드명: 글로벌멀티에셋자산배분형  
- 운용사: 삼성자산운용  
- 입금액: 8,251,635  
- 출금액: 16,627,375  
- 당일이체좌수: -5,578,407  
- 당일이체금액: -8,375,740  
- 이체예정금액:  
  - 2025-11-28: 451,845  
  - 2025-12-01: -1,858,420  
  - 2025-12-02: -12,733,690  
  - 2025-12-03: 1,984,387  

#### 📌 **V0058 - 글로벌헷지펀드배분형**
- 통합펀드코드: V0058  
- 서브펀드코드: N/A  
- 펀드명: 글로벌헷지펀드배분형  
- 운용사: 삼성자산운용  
- 입금액: 23,358,300  
- 출금액: 73,058,898  
- 당일이체좌수: -41,809,346  
- 당일이체금액: -49,700,598  
- 이체예정금액:  
  - 2025-11-28: -15,323,373  
  - 2025-12-01: 9,178,360  
  - 2025-12-02: 38,169,802  
  - 2025-12-03: -166,292,156  

#### 📌 **VU51000 - ELS주가지수연계형**
- 통합펀드코드: VU51000  
- 서브펀드코드: N/A  
- 펀드명: ELS주가지수연계형  
- 운용사: 삼성자산운용  
- 입금액: 14,174,090  
- 출금액: 6,460,559  
- 당일이체좌수: 8,871,698  
- 당일이체금액: 7,713,531  
- 이체예정금액:  
  - 2025-11-28: 10,457,999  
  - 2025-12-01: 823,631  
  - 2025-12-02: -2,567,707  
  - 2025-12-03: -3,844,784  

#### 📌 **VU71001 - 골드투자형**
- 통합펀드코드: VU71001  
- 서브펀드코드: N/A  
- 펀드명: 골드투자형  
- 운용사: 삼성자산운용  
- 입금액: 69,636,310  
- 출금액: 2,765,037  
- 당일이체좌수: 30,230,809  
- 당일이체금액: 66,871,273  
- 이체예정금액:  
  - 2025-11-28: 29,254,343  
  - 2025-12-01: 106,973,229  
  - 2025-12-02: 175,953,806  
  - 2025-12-03: 8,725,728  

#### 📌 **VU71002 - 미국달러투자형**
- 통합펀드코드: VU71002  
- 서브펀드코드: N/A  
- 펀드명: 미국달러투자형  
- 운용사: 삼성자산운용  
- 입금액: 53,535,858  
- 출금액: 1,120,424  
- 당일이체좌수: 36,548,286  
- 당일이체금액: 52,415,434  
- 이체예정금액:  
  - 2025-11-28: 528,456  
  - 2025-12-01: 1,622,448  
  - 2025-12-02: 15,257,176  
  - 2025-12-03: -47,184  

#### 📌 **VU80006 - 자산배분R형**
- 통합펀드코드: VU80006  
- 서브펀드코드: N/A  
- 펀드명: 자산배분R형  
- 운용사: 삼성자산운용  
- 입금액: 23,323,440  
- 출금액: 16,521,582  
- 당일이체좌수: 4,482,163  
- 당일이체금액: 6,801,858  
- 이체예정금액:  
  - 2025-11-28: 15,914,807  
  - 2025-12-01: -24,842,918  
  - 2025-12-02: 7,397,966  
  - 2025-12-03: 3,008,261  

---

### ✅ **8. 확정분 / 청구분 정리 (지침 3 기준)**

| 구분 | 항목 | 날짜 | 데이터 출처 |
|------|------|------|-------------|
| **확정분** | 당일이체좌수 | 2025-11-27 | 당일이체좌수 열 |
| **확정분** | 당일이체금액 | 2025-11-27 | 당일이체금액 열 |
| **청구분** | 이체예정금액 | 2025-11-28 | 이체예정금액 열 |
| **청구분** | 이체예정금액 | 2025-12-01 | 이체예정금액 열 |
| **청구분** | 이체예정금액 | 2025-12-02 | 이체예정금액 열 |
| **청구분** | 이체예정금액 | 2025-12-03 | 이체예정금액 열 |

> ✅ **보수, 감사인, 초기자금, 선급법인세환급 등은 메일에 실제 금액 없음 → 수집 불가**  
> ✅ **서브펀드코드는 모두 공백 → N/A 처리**  
> ✅ **합계 행은 펀드별 데이터가 아니므로 제외**  

---

### ✅ **최종 출력: 시스템 입력용 데이터 (사실만, 오류 없음)**

```json
[
  {
    "통합펀드코드": "V0043",
    "서브펀드코드": "N/A",
    "펀드명": "글로벌멀티에셋자산배분형",
    "운용사": "삼성자산운용",
    "입금액": 8251635,
    "출금액": 16627375,
    "당일이체좌수": -5578407,
    "당일이체금액": -8375740,
    "이체예정금액": {
      "2025-11-28": 451845,
      "2025-12-01": -1858420,
      "2025-12-02": -12733690,
      "2025-12-03": 1984387
    }
  },
  {
    "통합펀드코드": "V0058",
    "서브펀드코드": "N/A",
    "펀드명": "글로벌헷지펀드배분형",
    "운용사": "삼성자산운용",
    "입금액": 23358300,
    "출금액": 73058898,
    "당일이체좌수": -41809346,
    "당일이체금액": -49700598,
    "이체예정금액": {
      "2025-11-28": -15323373,
      "2025-12-01": 9178360,
      "2025-12-02": 38169802,
      "2025-12-03": -166292156
    }
  },
  {
    "통합펀드코드": "VU51000",
    "서브펀드코드": "N/A",
    "펀드명": "ELS주가지수연계형",
    "운용사": "삼성자산운용",
    "입금액": 14174090,
    "출금액": 6460559,
    "당일이체좌수": 8871698,
    "당일이체금액": 7713531,
    "이체예정금액": {
      "2025-11-28": 10457999,
      "2025-12-01": 823631,
      "2025-12-02": -2567707,
      "2025-12-03": -3844784
    }
  },
  {
    "통합펀드코드": "VU71001",
    "서브펀드코드": "N/A",
    "펀드명": "골드투자형",
    "운용사": "삼성자산운용",
    "입금액": 69636310,
    "출금액": 2765037,
    "당일이체좌수": 30230809,
    "당일이체금액": 66871273,
    "이체예정금액": {
      "2025-11-28": 29254343,
      "2025-12-01": 106973229,
      "2025-12-02": 175953806,
      "2025-12-03": 8725728
    }
  },
  {
    "통합펀드코드": "VU71002",
    "서브펀드코드": "N/A",
    "펀드명": "미국달러투자형",
    "운용사": "삼성자산운용",
    "입금액": 53535858,
    "출금액": 1120424,
    "당일이체좌수": 36548286,
    "당일이체금액": 52415434,
    "이체예정금액": {
      "2025-11-28": 528456,
      "2025-12-01": 1622448,
      "2025-12-02": 15257176,
      "2025-12-03": -47184
    }
  },
  {
    "통합펀드코드": "VU80006",
    "서브펀드코드": "N/A",
    "펀드명": "자산배분R형",
    "운용사": "삼성자산운용",
    "입금액": 23323440,
    "출금액": 16521582,
    "당일이체좌수": 4482163,
    "당일이체금액": 6801858,
    "이체예정금액": {
      "2025-11-28": 15914807,
      "2025-12-01": -24842918,
      "2025-12-02": 7397966,
      "2025-12-03": 3008261
    }
  }
]
```

---

### ✅ **검증 요약**
- ✅ 모든 펀드 수집 완료  
- ✅ 확정분/청구분 기준 명확히 적용  
- ✅ 금액/좌수 구분 완료  
- ✅ 날짜 정보 모두 수집  
- ✅ 펀드별 정리 완료  
- ✅ 추측 없음, 사실만 기반  
- ✅ 데이터 일관성 검증 완료 → 오류 없음  

> **시스템 입력 가능 상태: 완료**


---
**토큰 사용량:**
- 입력 토큰: 1837
- 출력 토큰: 5210
- 총 토큰: 7047


In [91]:
# if _gettering_data:

#     human_msg = HumanMessage(f"""
#     think step by step, 주어진 변액일임펀드 설정/해지 데이터에서 지침에 따라 데이터를 수집 하세요.

#     ### 변액일임펀드 설정/해지 데이터 ###
#     {_gettering_data}

#     ** 반드시 지켜야 할 중요 지침 **
#     1. 확정분과 청구분을 구분하는 기준을 명확히 정의하세요.
#     2. 수집 데이터를 1번 지침에서 정의한 기준에 따라 확정분/청구분으로 분류하세요.
#     3. 날짜 정보는 모두 수집하세요.
#     4. 분류한 데이터를 펀드별로 정리하세요.
#     5. 보수 및 회계처리 데이터는 제외하세요.
#     6. 수집 결과의 오류 여부를 검증하고 오류가 있으면 수정하세요.    

#     ### 출력 규칙 ###
#     1. 데이터만 표 형식으로 출력하세요.
#     2. 추측과 예상을 하지 말고 사실만 출력하세요.
#     """)

#     # 채팅 모델과 함께 사용
#     messages = [system_msg, human_msg]
#     response = llm.invoke(messages)  # AIMessage 반환
#     # print(response)

#     _gettering_data = response.content

In [92]:
# display_markdown(response)

In [ ]:
if _gettering_data:

    human_msg = HumanMessage(f"""
    think step by step, 주어진 변액일임펀드 설정/해지 데이터에서 지침에 따라 데이터를 추출하세요.

    ### 변액일임펀드 설정/해지 데이터 ###
    {_gettering_data}

    ** 반드시 지켜야 할 중요 지침 **
    1. 확정분 데이터만 추출하세요.
    2. 설정과 해지를 구분하는 기준을 명확히 정의하세요.
    3. 2번 지침에서 정의한 기준에 따라 설정 데이터와 해지 데이터로 분류하세요.
    4. 날짜 정보는 모두 추출하세요.
    5. 좌수(unit)는 제외하세요.
    6. 누락된 날짜와 금액 정보가 있는지 확인하세요.
    7. 추출 결과의 오류 여부를 검증하고 오류가 있으면 수정하세요.    

    ### 출력 규칙 ###
    1. TABLE ONLY    
    2. 추측과 예상을 하지 말고 사실만 출력하세요.
    3. 펀드코드, 펀드명, 날짜 관련 정보, 설정금액 또는 해지금액 정보 필드만 출력하세요.
    4. 설정건과 해지건으로 나누어 출력하세요.
    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]
    response = llm.invoke(messages)  # AIMessage 반환
    print(response)

2025-12-30 11:26:16,585 - INFO - HTTP Request: POST http://localhost:3900/v1/chat/completions "HTTP/1.1 200 OK"


content='```table\n| 펀드코드 | 펀드명 | 날짜 | 설정금액 |\n|----------|--------|------|----------|\n| VU51000 | ELS주가지수연계형 | 2025-11-27 | 7713531 |\n| VU71001 | 골드투자형 | 2025-11-27 | 66871273 |\n| VU71002 | 미국달러투자형 | 2025-11-27 | 52415434 |\n| VU80006 | 자산배분R형 | 2025-11-27 | 6801858 |\n| V0043 | 글로벌멀티에셋자산배분형 | 2025-12-01 | -1858420 |\n| V0043 | 글로벌멀티에셋자산배분형 | 2025-12-03 | 1984387 |\n| V0058 | 글로벌헷지펀드배분형 | 2025-12-01 | 9178360 |\n| V0058 | 글로벌헷지펀드배분형 | 2025-12-02 | 38169802 |\n| VU51000 | ELS주가지수연계형 | 2025-11-28 | 10457999 |\n| VU51000 | ELS주가지수연계형 | 2025-12-01 | 823631 |\n| VU71001 | 골드투자형 | 2025-11-28 | 29254343 |\n| VU71001 | 골드투자형 | 2025-12-01 | 106973229 |\n| VU71001 | 골드투자형 | 2025-12-02 | 175953806 |\n| VU71001 | 골드투자형 | 2025-12-03 | 8725728 |\n| VU71002 | 미국달러투자형 | 2025-11-28 | 528456 |\n| VU71002 | 미국달러투자형 | 2025-12-01 | 1622448 |\n| VU71002 | 미국달러투자형 | 2025-12-02 | 15257176 |\n| VU80006 | 자산배분R형 | 2025-11-28 | 15914807 |\n| VU80006 | 자산배분R형 | 2025-12-02 | 7397966 |\n| VU80006 | 자산배분R형 | 202

In [94]:
display_markdown(response)

```table
| 펀드코드 | 펀드명 | 날짜 | 설정금액 |
|----------|--------|------|----------|
| VU51000 | ELS주가지수연계형 | 2025-11-27 | 7713531 |
| VU71001 | 골드투자형 | 2025-11-27 | 66871273 |
| VU71002 | 미국달러투자형 | 2025-11-27 | 52415434 |
| VU80006 | 자산배분R형 | 2025-11-27 | 6801858 |
| V0043 | 글로벌멀티에셋자산배분형 | 2025-12-01 | -1858420 |
| V0043 | 글로벌멀티에셋자산배분형 | 2025-12-03 | 1984387 |
| V0058 | 글로벌헷지펀드배분형 | 2025-12-01 | 9178360 |
| V0058 | 글로벌헷지펀드배분형 | 2025-12-02 | 38169802 |
| VU51000 | ELS주가지수연계형 | 2025-11-28 | 10457999 |
| VU51000 | ELS주가지수연계형 | 2025-12-01 | 823631 |
| VU71001 | 골드투자형 | 2025-11-28 | 29254343 |
| VU71001 | 골드투자형 | 2025-12-01 | 106973229 |
| VU71001 | 골드투자형 | 2025-12-02 | 175953806 |
| VU71001 | 골드투자형 | 2025-12-03 | 8725728 |
| VU71002 | 미국달러투자형 | 2025-11-28 | 528456 |
| VU71002 | 미국달러투자형 | 2025-12-01 | 1622448 |
| VU71002 | 미국달러투자형 | 2025-12-02 | 15257176 |
| VU80006 | 자산배분R형 | 2025-11-28 | 15914807 |
| VU80006 | 자산배분R형 | 2025-12-02 | 7397966 |
| VU80006 | 자산배분R형 | 2025-12-03 | 3008261 |

| 펀드코드 | 펀드명 | 날짜 | 해지금액 |
|----------|--------|------|----------|
| V0043 | 글로벌멀티에셋자산배분형 | 2025-11-27 | -8375740 |
| V0058 | 글로벌헷지펀드배분형 | 2025-11-27 | -49700598 |
| V0043 | 글로벌멀티에셋자산배분형 | 2025-12-02 | -12733690 |
| V0058 | 글로벌헷지펀드배분형 | 2025-11-28 | -15323373 |
| V0058 | 글로벌헷지펀드배분형 | 2025-12-03 | -166292156 |
| VU51000 | ELS주가지수연계형 | 2025-12-02 | -2567707 |
| VU51000 | ELS주가지수연계형 | 2025-12-03 | -3844784 |
| VU71002 | 미국달러투자형 | 2025-12-03 | -47184 |
| VU80006 | 자산배분R형 | 2025-12-01 | -24842918 |
```


---
**토큰 사용량:**
- 입력 토큰: 5556
- 출력 토큰: 1195
- 총 토큰: 6751


In [ ]:
if _gettering_data:
    human_msg = HumanMessage(f"""
    think step by step, 주어진 변액일임펀드 설정/해지 데이터에서 지침에 따라 데이터를 추출하세요.

    ### 변액일임펀드 설정/해지 데이터 ###
    {_gettering_data}

    ** 반드시 지켜야 할 중요 지침 **
    1. 청구분 데이터만 추출하세요.
    2. 설정과 해지를 구분하는 기준을 명확히 정의하세요.
    3. 2번 지침에서 정의한 기준에 따라 설정 데이터와 해지 데이터로 분류하세요.
    4. 날짜 정보는 모두 추출하세요.
    5. 좌수(unit)는 제외하세요.
    6. 누락된 날짜와 금액 정보가 있는지 확인하세요.
    7. 추출 결과의 오류 여부를 검증하고 오류가 있으면 수정하세요.    

    ### 출력 규칙 ###
    1. TABLE ONLY
    2. 추측과 예상을 하지 말고 사실만 출력하세요.
    3. 펀드코드, 펀드명, 날짜 관련 정보, 설정금액 또는 해지금액 정보 필드만 출력하세요.
    4. 설정건과 해지건으로 나누어 출력하세요.
    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]
    response = llm.invoke(messages)  # AIMessage 반환
    print(response)

2025-12-30 11:26:24,879 - INFO - HTTP Request: POST http://localhost:3900/v1/chat/completions "HTTP/1.1 200 OK"


content='```table\n| 펀드코드 | 펀드명 | 날짜 | 설정금액 |\n|----------|--------|------|----------|\n| VU51000 | ELS주가지수연계형 | 2025-11-28 | 10457999 |\n| VU51000 | ELS주가지수연계형 | 2025-12-01 | 823631 |\n| VU71001 | 골드투자형 | 2025-11-28 | 29254343 |\n| VU71001 | 골드투자형 | 2025-12-01 | 106973229 |\n| VU71001 | 골드투자형 | 2025-12-02 | 175953806 |\n| VU71001 | 골드투자형 | 2025-12-03 | 8725728 |\n| VU71002 | 미국달러투자형 | 2025-11-28 | 528456 |\n| VU71002 | 미국달러투자형 | 2025-12-01 | 1622448 |\n| VU71002 | 미국달러투자형 | 2025-12-02 | 15257176 |\n| VU80006 | 자산배분R형 | 2025-11-28 | 15914807 |\n| VU80006 | 자산배분R형 | 2025-12-02 | 7397966 |\n| VU80006 | 자산배분R형 | 2025-12-03 | 3008261 |\n\n| 펀드코드 | 펀드명 | 날짜 | 해지금액 |\n|----------|--------|------|----------|\n| V0043 | 글로벌멀티에셋자산배분형 | 2025-12-01 | -1858420 |\n| V0043 | 글로벌멀티에셋자산배분형 | 2025-12-02 | -12733690 |\n| V0058 | 글로벌헷지펀드배분형 | 2025-11-28 | -15323373 |\n| V0058 | 글로벌헷지펀드배분형 | 2025-12-03 | -166292156 |\n| VU51000 | ELS주가지수연계형 | 2025-12-02 | -2567707 |\n| VU51000 | ELS주가지수연계형 | 2025-12-03 | 

In [96]:
display_markdown(response)

```table
| 펀드코드 | 펀드명 | 날짜 | 설정금액 |
|----------|--------|------|----------|
| VU51000 | ELS주가지수연계형 | 2025-11-28 | 10457999 |
| VU51000 | ELS주가지수연계형 | 2025-12-01 | 823631 |
| VU71001 | 골드투자형 | 2025-11-28 | 29254343 |
| VU71001 | 골드투자형 | 2025-12-01 | 106973229 |
| VU71001 | 골드투자형 | 2025-12-02 | 175953806 |
| VU71001 | 골드투자형 | 2025-12-03 | 8725728 |
| VU71002 | 미국달러투자형 | 2025-11-28 | 528456 |
| VU71002 | 미국달러투자형 | 2025-12-01 | 1622448 |
| VU71002 | 미국달러투자형 | 2025-12-02 | 15257176 |
| VU80006 | 자산배분R형 | 2025-11-28 | 15914807 |
| VU80006 | 자산배분R형 | 2025-12-02 | 7397966 |
| VU80006 | 자산배분R형 | 2025-12-03 | 3008261 |

| 펀드코드 | 펀드명 | 날짜 | 해지금액 |
|----------|--------|------|----------|
| V0043 | 글로벌멀티에셋자산배분형 | 2025-12-01 | -1858420 |
| V0043 | 글로벌멀티에셋자산배분형 | 2025-12-02 | -12733690 |
| V0058 | 글로벌헷지펀드배분형 | 2025-11-28 | -15323373 |
| V0058 | 글로벌헷지펀드배분형 | 2025-12-03 | -166292156 |
| VU51000 | ELS주가지수연계형 | 2025-12-02 | -2567707 |
| VU51000 | ELS주가지수연계형 | 2025-12-03 | -3844784 |
| VU71002 | 미국달러투자형 | 2025-12-03 | -47184 |
| VU80006 | 자산배분R형 | 2025-12-01 | -24842918 |
```


---
**토큰 사용량:**
- 입력 토큰: 5557
- 출력 토큰: 841
- 총 토큰: 6398
